# Setup

In [22]:
from sentence_transformers import SentenceTransformer

import math 
import numpy as np
import scipy


# Obtain Vector Embeddings

In [9]:
# Example documents
documents = [
    'Bugs introduced by the intern had to be squashed by the lead developer.',
    'Bugs found by the quality assurance engineer were difficult to debug.',
    'Bugs are common throughout the warm summer months, according to the entomologist.',
    'Bugs, in particular spiders, are extensively studied by arachnologists.'
]

# Load a pre-trained model
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

# Generate embeddings
embeddings = model.encode(documents)

embeddings.shape  # Should be (4, 384) for this model

embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13173.95it/s]


array([[-0.22804345, -0.2464771 , -0.0031926 , ...,  0.4552817 ,
         0.63419735,  0.537505  ],
       [-0.357916  , -0.3208398 ,  0.15963255, ..., -0.07050646,
         0.927502  ,  0.3437727 ],
       [ 0.20302923, -0.26898587,  0.16285145, ..., -0.19651009,
        -0.03379817,  0.59561545],
       [-0.04264311, -0.45721602, -0.09526499, ..., -0.5803075 ,
         0.17248413,  0.09127872]], shape=(4, 384), dtype=float32)

## L2 (Euclidean) Distance

In [19]:
def euclidean_distance_fn(vector1, vector2):
    squared_sum = sum((x - y) ** 2 for x, y in zip(vector1, vector2))
    return math.sqrt(squared_sum)

In [ ]:
print("euclidean_distance_fn(embeddings[0], embeddings[1]):", euclidean_distance_fn(embeddings[0], embeddings[1]))
print("euclidean_distance_fn(embeddings[1], embeddings[0]):", euclidean_distance_fn(embeddings[1], embeddings[0]))

l2_dist_manual = np.zeros([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        l2_dist_manual[i,j] = euclidean_distance_fn(embeddings[i], embeddings[j])

print("l2_dist_manual:\n", l2_dist_manual)



###  Make the manual calculation more efficient

In [20]:
l2_dist_manual_improved = np.zeros([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        if j > i: # Calculate the upper triangle only
            l2_dist_manual_improved[i,j] = euclidean_distance_fn(embeddings[i], embeddings[j])
        elif i > j: # Copy the uper triangle to the lower triangle
            l2_dist_manual_improved[i,j] = l2_dist_manual[j,i]

print("l2_dist_manual_improved:\n", l2_dist_manual_improved)

l2_dist_manual_improved:
 [[0.         5.96178921 7.33939963 7.15578303]
 [5.96178921 0.         7.76861552 7.39358971]
 [7.33939963 7.76861552 0.         5.91992767]
 [7.15578303 7.39358971 5.91992767 0.        ]]


### Calculate L2 distance using `scipy`

In [23]:
l2_dist_scipy = scipy.spatial.distance.cdist(embeddings, embeddings, 'euclidean')
print("l2_dist_scipy:\n", l2_dist_scipy)

l2_dist_scipy:
 [[0.         5.96178991 7.33940016 7.15578184]
 [5.96178991 0.         7.76861592 7.39359042]
 [7.33940016 7.76861592 0.         5.91992798]
 [7.15578184 7.39359042 5.91992798 0.        ]]


## 